# GPFA Latent Dynamics Analysis

This notebook fits **Gaussian Process Factor Analysis (GPFA)** to single-unit spiking
data from one recording session, in order to extract low-dimensional latent
trajectories that summarize population-level neural dynamics on each trial.

**Pipeline overview**

1. Load session data (spikes, trial times) and analysis parameters
2. Select well-isolated units and filter out units with too few spikes
3. Build per-trial spike trains for GPFA
4. Cross-validate to choose the latent dimensionality (`x_dim`) via leave-neuron-out
   prediction error
5. Fit the final GPFA model at the selected dimensionality and save results
6. Visualize latent trajectories (single-trial, trial-averaged, 3D, alongside spike rasters)
7. Relate latent dimensions to behavioral variables

**Requirements:** `neo`, `quantities`, `elephant`, `scikit-learn`, `numpy`, `matplotlib`,
plus the local `data_paths` and `Analysis_library` modules from this repo (see the
README for setup instructions).

## 1. Setup

Import dependencies and add the project root to the path so the local
`Analysis_library` package and `data_paths` module can be imported.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import neo
import quantities as pq
from elephant.gpfa import GPFA

notebook_dir = Path.cwd()
project_root = notebook_dir.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from data_paths import DATA_PATH
from Analysis_library.file_loading import SessionData
import Analysis_library.analysis as ana_lib

## 2. Load Session Data

Load the recording session specified by `SESSION_KEY`. `SessionData` wraps the raw
`.mat` file and exposes spikes and trial timing.

In [ ]:
SESSION_KEY = "332"
sessions = [SessionData(DATA_PATH[SESSION_KEY])]
s = sessions[0]

print("Session:", s.mat_file)
print("Number of units:", len(s.units_dict["spikes"]))
print("Number of trials:", len(s.trials_dict["t"]))

## 3. Analysis Parameters

Core parameters for binning and trial windowing:

- `SAMPLE_RATE_HZ` – sampling rate of the raw spike times
- `BIN_SIZE_MS` – GPFA time-bin width
- `PRE_TRIAL_S` / `POST_TRIAL_S` – window (in seconds) around each trial event used
  to build each trial's spike trains

In [ ]:
SAMPLE_RATE_HZ = 30000.0
BIN_SIZE_MS = 30 # Consider using a smaller bin size (e.g., 10 ms) for better temporal resolution, but this may increase computational load.
PRE_TRIAL_S = 1.0
POST_TRIAL_S = 4.0

## 4. Unit Selection

Identify well-isolated ("good") units and pull the trial event times that define
each trial window.

In [ ]:
good_units = ana_lib.get_good_units(s.units_dict, verbose=False)
good_units = np.asarray(good_units, dtype=int)
print("Number of good units:", len(good_units))
trial_times = np.asarray(s.trials_dict["t"], dtype=float)

print("\nFirst 3 trial times:")
print(trial_times[:3])
# shape: trial x neuron x time bins

### 4.1 Filter units by minimum spike count

GPFA fitting is unstable for units with very few spikes across all trial windows,
so units with fewer than `MIN_SPIKES` total spikes are dropped before building the
GPFA dataset.

In [ ]:
spikes = s.units_dict["spikes"]
TRIAL_DURATION_S = PRE_TRIAL_S + POST_TRIAL_S

# Count spikes for each unit across all trial windows
spike_counts_per_unit = []

for unit_idx in good_units:
    total_spikes = 0

    for trial_time in trial_times:
        trial_start = trial_time - PRE_TRIAL_S
        trial_stop = trial_time + POST_TRIAL_S
        spike_samples = np.asarray(spikes[unit_idx], dtype=float).squeeze()
        spike_times_s = (spike_samples / SAMPLE_RATE_HZ)
        trial_spikes = spike_times_s[(spike_times_s >= trial_start) & (spike_times_s < trial_stop)]
        total_spikes += len(trial_spikes)
    spike_counts_per_unit.append(total_spikes)
spike_counts_per_unit = np.asarray(spike_counts_per_unit)

# Keep units with at least 5 spikes across trial windows
MIN_SPIKES = 5
gpfa_units = good_units[spike_counts_per_unit >= MIN_SPIKES]

print("Original good units:", len(good_units))
print("Units kept for GPFA:", len(gpfa_units))

## 5. Build Per-Trial Spike Trains

For each trial, convert each selected unit's spike times (in samples) into a
`neo.SpikeTrain` aligned to trial onset (`t=0` at `trial_time - PRE_TRIAL_S`),
spanning the full trial duration (`PRE_TRIAL_S + POST_TRIAL_S`). The result,
`gpfa_data`, is a list of trials, each a list of spike trains (one per neuron) —
the format `elephant.gpfa.GPFA` expects.

In [ ]:
# Build GPFA trials

gpfa_data = []

for trial_time in trial_times:
    trial_spiketrains = []
    trial_start = trial_time - PRE_TRIAL_S
    trial_stop = trial_time + POST_TRIAL_S

    for unit_idx in gpfa_units:
        spike_samples = np.asarray(spikes[unit_idx], dtype=float).squeeze()
        spike_times_s = (spike_samples / SAMPLE_RATE_HZ)
        trial_spikes = spike_times_s[(spike_times_s >= trial_start) & (spike_times_s < trial_stop)]
        trial_spikes = (trial_spikes - trial_start)
        spike_train = neo.SpikeTrain(
            trial_spikes * pq.s,
            t_start=0 * pq.s,
            t_stop=TRIAL_DURATION_S * pq.s
        )
        trial_spiketrains.append(spike_train)
    
    gpfa_data.append(trial_spiketrains)

print("Number of neurons per trial:", len(gpfa_data[0]))
print(s.data.keys())

## 6. Cross-Validation Setup

We select the GPFA latent dimensionality (`x_dim`) by cross-validated leave-neuron-out
prediction: for each candidate dimensionality, fit GPFA on a subset of neurons and
trials, then check how well the fitted model predicts the activity of held-out
neurons on held-out trials. Lower prediction error indicates a better dimensionality.

- `X_DIMS` – candidate latent dimensionalities to evaluate
- `N_FOLDS` – number of cross-validation folds over trials
- `RANDOM_SEED` – fixed seed for reproducible splits

In [ ]:
from sklearn.model_selection import KFold
from elephant.conversion import BinnedSpikeTrain
import numpy as np
import time

X_DIMS = [1, 2, 3, 4, 5, 8, 10, 15] # [2, 4, 6, 8]
N_FOLDS = 4
RANDOM_SEED = 42

### 6.1 Bin spike trains

Convert each trial's spike trains into binned spike counts
(`trials x neurons x time bins`), using `BIN_SIZE_MS` from Section 3.

In [ ]:
binned_trials = []

# trials x neurons x time bins
for trial in gpfa_data:
    binned = BinnedSpikeTrain(trial, bin_size=BIN_SIZE_MS * pq.ms)
    counts = binned.to_array()
    binned_trials.append(counts)
binned_trials = np.asarray(binned_trials)

print( "Binned data shape:", binned_trials.shape)

### 6.2 Train/test split helpers

- `make_trial_split` — K-fold split over trials (for cross-validation folds)
- `make_neuron_split` — splits neurons into a "fit" set (used to fit GPFA) and a
  held-out set (used only to evaluate prediction quality)

In [ ]:
def make_trial_split(n_trials, n_folds=4, seed=RANDOM_SEED):
    """Train/test split by holding out whole trials."""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return list(kf.split(np.arange(n_trials)))

def make_neuron_split(n_neurons, test_frac=0.3, seed=RANDOM_SEED):
    """Split neurons into a fit set and a held-out (predicted) set."""
    rng = np.random.default_rng(seed)
    n_test = int(round(n_neurons * test_frac))
    test_idx = rng.choice(n_neurons, size=n_test, replace=False)
    train_idx = np.setdiff1d(np.arange(n_neurons), test_idx)
    return train_idx, test_idx

TEST_NEURON_FRAC = 0.3

fit_neurons, heldout_neurons = make_neuron_split(
    binned_trials.shape[1], test_frac=TEST_NEURON_FRAC
)
print(f"Held out {len(heldout_neurons)} / {binned_trials.shape[1]} neurons")

### 6.3 Held-out neuron prediction

Given a fitted GPFA model, predict a held-out neuron's activity from the latent
trajectories inferred from the *other* neurons. This is the core function used to
score each candidate `x_dim` during cross-validation.

> **Note:** Elephant's GPFA implementation can silently drop neurons with no
> training spikes when fitting. This function guards against the resulting
> neuron-count mismatch by falling back to the neuron's mean firing rate if it
> was dropped from the fitted model.

In [ ]:
from elephant.gpfa import gpfa_core
import copy
import numpy as np

# NOTE: This updated function handles cases where GPFA drops neurons with no training spikes, preventing neuron-count mismatches between the fitted model and the original data.

def predict_heldout_neuron_gpfa(counts, params, neuron_idx):
    # Number of neurons in fitted GPFA model
    n_model_neurons = params["C"].shape[0]

    # If Elephant dropped the held-out neuron during fitting,
    # return its mean firing instead of crashing.
    if neuron_idx >= n_model_neurons:
        return np.full(
            counts.shape[1],
            np.mean(counts[neuron_idx])
        )

    observed_idx = np.ones(n_model_neurons, dtype=bool)
    observed_idx[neuron_idx] = False
    params_obs = copy.deepcopy(params)

    # Remove held-out neuron from observation parameters
    params_obs["C"] = params["C"][observed_idx, :]
    params_obs["d"] = params["d"][observed_idx]
    params_obs["R"] = params["R"][np.ix_(observed_idx, observed_idx)]

    # Create Elephant sequence
    seq = np.zeros(1, dtype=[("T", int), ("y", object)])
    seq["T"][0] = counts.shape[1]
    # Only use neurons that actually exist in the fitted model
    seq["y"][0] = counts[:n_model_neurons, :][observed_idx, :]

    # GP-aware latent inference
    inferred_seq, _ = gpfa_core.exact_inference_with_ll(seq, params_obs, get_ll=False)
    latent_mean = inferred_seq["latent_variable"][0]

    # Predict held-out neuron
    prediction = (params["C"][neuron_idx] @ latent_mean + params["d"][neuron_idx])
    return prediction

## 7. Cross-Validate Latent Dimensionality

For each candidate `x_dim`, fit GPFA across `N_FOLDS` cross-validation folds (drawn
only from the training trials — a fully held-out test set is set aside up front and
never touched during model selection) and record the leave-neuron-out prediction
error on both the training and validation folds. The best `x_dim` is the one with
the lowest mean validation error.

A final model at the best `x_dim` is then fit on all training trials and scored once
on the untouched test trials, giving an unbiased estimate of prediction performance.

**Note:** this cell can take a while to run (it fits `len(X_DIMS) * N_FOLDS`
GPFA models plus one final model).

In [ ]:
# FIXED trial cross validation

from sklearn.model_selection import train_test_split, KFold

# Hold out completely unseen trials for the final evaluation.
# These trials are NEVER used during model selection or cross-validation.
train_trial_idx, test_trial_idx = train_test_split(
    np.arange(len(gpfa_data)),
    test_size=0.2,
    random_state=RANDOM_SEED,
    shuffle=True
)

train_gpfa_data = [gpfa_data[i] for i in train_trial_idx]

# Perform K-fold cross-validation only on the training trials
kf = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED
)

mean_prediction_errors = []
all_fold_errors = {}
training_fold_errors = {}

# Cross-validation loop
for x_dim in X_DIMS:
    print(f"\nTesting x_dim = {x_dim}")
    start_time = time.time()

    training_errors = []
    validation_errors = []

    for fold_idx, (train_idx, validation_idx) in enumerate(kf.split(train_gpfa_data)):
        print(f"  Fold {fold_idx + 1}/{N_FOLDS}")

        # Training trials for this fold
        fold_train_trials = [train_gpfa_data[i] for i in train_idx]

        # Fit GPFA
        gpfa_cv = GPFA(
            bin_size=BIN_SIZE_MS * pq.ms,
            x_dim=x_dim,
            em_max_iters=100,
            verbose=False
        )

        gpfa_cv.fit(fold_train_trials)
        params = gpfa_cv.params_estimated

        # Training error
        train_neuron_errors = []
        for neuron_idx in heldout_neurons:
            trial_errors = []
            for trial_idx in train_trial_idx[train_idx]:
                counts = binned_trials[trial_idx]
                prediction = predict_heldout_neuron_gpfa(counts, params, neuron_idx)
                actual = counts[neuron_idx]
                error = np.sum((actual - prediction) ** 2)
                trial_errors.append(error)
            train_neuron_errors.append(np.mean(trial_errors))
        train_error = np.mean(train_neuron_errors)
        training_errors.append(train_error)

        # Validation error
        validation_neuron_errors = []
        for neuron_idx in heldout_neurons:
            trial_errors = []
            for trial_idx in train_trial_idx[validation_idx]:
                counts = binned_trials[trial_idx]
                prediction = predict_heldout_neuron_gpfa(counts, params, neuron_idx)
                actual = counts[neuron_idx]
                error = np.sum((actual - prediction) ** 2)
                trial_errors.append(error)
            validation_neuron_errors.append(np.mean(trial_errors))
        validation_error = np.mean(validation_neuron_errors)
        validation_errors.append(validation_error)

        print(
            f"    Train error: {train_error:.3f} | "
            f"Validation error: {validation_error:.3f}"
        )

    mean_prediction_errors.append(np.mean(validation_errors))
    all_fold_errors[x_dim] = validation_errors
    training_fold_errors[x_dim] = training_errors

    print("Mean train error:", np.mean(training_errors))
    print("Mean validation error:", np.mean(validation_errors))
    print("Time:", round((time.time() - start_time) / 60, 2), "minutes")

# Select best dimensionality
best_idx = np.argmin(mean_prediction_errors)
best_x_dim = X_DIMS[best_idx]

print(f"\nBest x_dim: {best_x_dim}")
print(f"Mean CV validation error: {mean_prediction_errors[best_idx]:.3f}")

# FINAL EVALUATION ON THE COMPLETELY UNSEEN TEST SET
print("\nFinal evaluation on held-out test set...")
final_gpfa = GPFA(
    bin_size=BIN_SIZE_MS * pq.ms,
    x_dim=best_x_dim,
    em_max_iters=100,
    verbose=False
)

# Train on ALL training trials
final_gpfa.fit(train_gpfa_data)
final_params = final_gpfa.params_estimated

test_neuron_errors = []
for neuron_idx in heldout_neurons:
    trial_errors = []
    for trial_idx in test_trial_idx:
        counts = binned_trials[trial_idx]
        prediction = predict_heldout_neuron_gpfa(counts, final_params, neuron_idx)
        actual = counts[neuron_idx]
        error = np.sum((actual - prediction) ** 2)
        trial_errors.append(error)
    test_neuron_errors.append(np.mean(trial_errors))
final_test_error = np.mean(test_neuron_errors)
print(f"\nFinal held-out test error: {final_test_error:.3f}")

## 8. Save Cross-Validation Results

Save the cross-validation model, selected dimensionality, and CV errors to disk
so they can be reloaded without repeating the fit above.

In [ ]:
# Save GPFA results model
import pickle
from pathlib import Path

SAVE_DIR = Path("ISS")
SAVE_DIR.mkdir(exist_ok=True)

results = {
    # Fitted model
    "model": final_gpfa,
    "params": final_gpfa.params_estimated,

    # Best model information
    "x_dim": best_x_dim,
    "candidate_dims": X_DIMS,
    "cv_errors": all_fold_errors,
    "mean_cv_errors": mean_prediction_errors,

    # Dataset information
    "session": SESSION_KEY,
    "n_trials": len(gpfa_data),
    "n_neurons": binned_trials.shape[1],
    "good_units": good_units,
    "gpfa_units": gpfa_units,
    "heldout_neurons": heldout_neurons,

    # Analysis settings
    "bin_size_ms": BIN_SIZE_MS,
    "pre_trial_s": PRE_TRIAL_S,
    "post_trial_s": POST_TRIAL_S,
    "n_folds": N_FOLDS,
    "random_seed": RANDOM_SEED,
}
save_path = SAVE_DIR / "gpfa_results.pkl"
with open(save_path, "wb") as f:
    pickle.dump(results, f)
print("Saved GPFA results.")
print(save_path.resolve())

## 9. Visualize Cross-Validation Results

Plot mean validation prediction error (± SEM across folds) as a function of
candidate latent dimensionality, and report the best-performing `x_dim`.

In [ ]:
print("CV RESULTS")

# Only include completed x_dims
plot_dims = [d for d in X_DIMS if d != 20 and d != 4]

# Compute mean ± SEM across folds
mean_errors = [np.mean(all_fold_errors[d]) for d in plot_dims]
sem_errors = [
    np.std(all_fold_errors[d], ddof=1) / np.sqrt(len(all_fold_errors[d]))
    for d in plot_dims
]

# Print results
for d, mean_error, sem_error in zip(plot_dims, mean_errors, sem_errors):
    print(
        f"x_dim = {d}: "
        f"mean validation error = {mean_error:.4f} ± {sem_error:.4f}"
    )

# Best dimensionality
best_index = np.argmin(mean_errors)
best_x_dim = plot_dims[best_index]

print(f"\nBest GPFA dimensionality: {best_x_dim}")

# Plot
plt.figure(figsize=(8, 5))
plt.errorbar(
    plot_dims,
    mean_errors,
    yerr=sem_errors,
    marker="o",
    capsize=4
)
plt.xlabel("Latent dimensionality")
plt.ylabel("Mean validation prediction error")
plt.title("GPFA Dimensionality Selection (5-Fold Cross-Validation)")
plt.xticks(plot_dims)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("X_DIMS:", X_DIMS)
print("Completed x_dims:", sorted(all_fold_errors.keys()))

### 9.1 Training vs. validation error

Comparing training and validation error side-by-side helps flag overfitting —
i.e. dimensionalities where training error keeps dropping but validation error
does not.

In [ ]:
# Plot training vs validation error
plot_dims = [d for d in X_DIMS if d != 20 and d != 4]

# Training mean ± SEM
mean_train_errors = [np.mean(training_fold_errors[d]) for d in plot_dims]
sem_train_errors = [
    np.std(training_fold_errors[d], ddof=1) /
    np.sqrt(len(training_fold_errors[d]))
    for d in plot_dims
]

# Validation mean ± SEM
mean_val_errors = [np.mean(all_fold_errors[d]) for d in plot_dims]
sem_val_errors = [
    np.std(all_fold_errors[d], ddof=1) /
    np.sqrt(len(all_fold_errors[d]))
    for d in plot_dims
]

# Print values
for d, tr, va in zip(plot_dims, mean_train_errors, mean_val_errors):
    print(f"x_dim={d}: " f"train={tr:.4f}, validation={va:.4f}")

plt.figure(figsize=(8,5))
plt.errorbar(plot_dims,mean_train_errors, yerr=sem_train_errors, marker="o", capsize=4, label="Training error")
plt.errorbar(plot_dims, mean_val_errors,  yerr=sem_val_errors,  marker="o", capsize=4, label="Validation error")
plt.xlabel("Latent dimensionality (x_dim)")
plt.ylabel("Prediction error")
plt.title("GPFA Training vs Validation Error")
plt.xticks(plot_dims)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Fit Final GPFA Model

Refit GPFA at `best_x_dim` using **all** trials (not just the training split) to
get the final model and latent trajectories used for the analyses below.

> **TODO (from original notebook):** figure out how to persist this model so it
> can be reused for inference on new data without refitting.

In [ ]:
# Fit final GPFA model using best x_dim from cross-validation

# NOTE: figure out how to save the model so that it can be used later for inference on new data each time
final_gpfa = GPFA(
    bin_size=BIN_SIZE_MS * pq.ms,
    x_dim= best_x_dim,
    em_max_iters=100,
    verbose=False
)
final_gpfa.fit(gpfa_data)
print("Final GPFA model fitted!")

# Extract latent trajectories
latent_trajectories = final_gpfa.transform(gpfa_data)

print("Number of trials:", len(latent_trajectories))
print("First trial latent shape:", latent_trajectories[0].shape)

### 10.1 Save final results

Overwrites the saved results from Section 8 with the final model (fit on all
trials) and its latent trajectories.

In [ ]:
import pickle
from pathlib import Path

SAVE_DIR = Path("ISS")
SAVE_DIR.mkdir(exist_ok=True)

results = {
    "model": final_gpfa,
    "params": final_gpfa.params_estimated,
    "x_dim": best_x_dim,
    "session": SESSION_KEY,
    "bin_size_ms": BIN_SIZE_MS,
    "pre_trial_s": PRE_TRIAL_S,
    "post_trial_s": POST_TRIAL_S,
    "good_units": good_units,
    "gpfa_units": gpfa_units,
    "n_trials": len(gpfa_data),
    "n_neurons": binned_trials.shape[1],
    "candidate_dims": X_DIMS,
    "cv_errors": all_fold_errors,
    "heldout_neurons": heldout_neurons,
    "random_seed": RANDOM_SEED,
}

with open(SAVE_DIR / "gpfa_results.pkl", "wb") as f:
    pickle.dump(results, f)

print("Saved GPFA results.")
print((SAVE_DIR / "gpfa_results.pkl").resolve())

## 11. Visualize Latent Trajectories

### 11.1 Single-trial latent trajectories

Plot every latent dimension over time for one example trial.

> **TODO (from original notebook):** add raster plots alongside the latent
> trajectory plots for each trial, and a cumulative-explained-variance plot for
> the latent dimensions.

In [ ]:
# Inspect and plot GPFA latent trajectory - TO-DO: Add raster plots alongside of latent trajectory plots for ea trial
# plot the cumulative explained variance for each latent dimension to visualize how much variance is captured by the selected number of dimensions PCA
trial_idx = 0
latent = latent_trajectories[trial_idx]
print("Latent trajectory shape:", latent.shape)
plt.figure(figsize=(10, 5))
for dim in range(latent.shape[0]):
    plt.plot(latent[dim], label=f"Latent {dim+1}")
plt.xlabel("Time bin")
plt.ylabel("Latent activity")
plt.title(f"GPFA latent trajectories - Trial {trial_idx}")
plt.legend()
plt.tight_layout()
plt.show()

### 11.2 3D trajectory (first 3 latent dimensions)

Visualize the same example trial as a trajectory through the first three latent
dimensions.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

trial_idx = 0
latent = latent_trajectories[trial_idx]
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

ax.plot(
    latent[0],
    latent[1],
    latent[2]
)

ax.set_xlabel("Latent 1")
ax.set_ylabel("Latent 2")
ax.set_zlabel("Latent 3")
ax.set_title(f"GPFA neural trajectory - Trial {trial_idx}")
plt.show()

### 11.3 Trial-averaged latent dynamics

Average the latent trajectories across all trials and plot mean ± SEM for each
latent dimension.

> **TODO (from original notebook):** shade the ± SEM band directly around each
> latent curve (currently implemented — kept as noted in the original for
> reference).

In [ ]:
# Average GPFA latent trajectories across trials - TO-DO: need shaded STD / error bars cloud around each latent curve

latent_array = np.stack(latent_trajectories, axis=0)
print("All latent trajectories shape:", latent_array.shape)

mean_latent = np.mean(latent_array, axis=0)
sem_latent = (
    np.std(latent_array, axis=0, ddof=1)
    / np.sqrt(latent_array.shape[0])
)
plt.figure(figsize=(10, 5))
for dim in range(mean_latent.shape[0]):
    plt.plot(
        mean_latent[dim],
        label=f"Latent {dim+1}"
    )
    plt.fill_between(
        np.arange(mean_latent.shape[1]),
        mean_latent[dim] - sem_latent[dim],
        mean_latent[dim] + sem_latent[dim],
        alpha=0.25
    )

plt.xlabel("Time bin")
plt.ylabel("Latent activity")
plt.title("Trial-averaged GPFA latent dynamics (mean ± SEM)")
plt.legend()
plt.tight_layout()
plt.show()

### 11.4 Latent trajectories with spike raster

Stack the per-dimension latent traces above a spike raster for the same trial, to
visually compare latent dynamics against raw population spiking.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

trial_idx = 0

latent = latent_trajectories[trial_idx]   # (15,166)
trial_spike_counts = binned_trials[trial_idx]  # (152,166)
n_latents = latent.shape[0]
n_neurons = trial_spike_counts.shape[0]
time_bins = np.arange(latent.shape[1])

fig, axes = plt.subplots(n_latents + 1, 1, figsize=(12, 18), sharex=True, gridspec_kw={"height_ratios": [1]*n_latents + [3]})
# Plot latent dimensions
for dim in range(n_latents):
    axes[dim].plot(time_bins, latent[dim])
    axes[dim].set_ylabel(f"L{dim+1}", rotation=0, labelpad=15)
    axes[dim].grid(alpha=0.3)

# Raster plot
axes[-1].imshow(trial_spike_counts, aspect="auto", interpolation="nearest", cmap="binary")
axes[-1].set_ylabel("Neuron")
axes[-1].set_xlabel("Time bin")

plt.suptitle(f"GPFA latent trajectories + spike raster | Trial {trial_idx}", y=0.995)
plt.tight_layout()
plt.show()

### 11.5 GPFA loading matrix

Visualize the loading matrix `C` from the final fitted model (each row is a
neuron's contribution to each latent dimension), with neurons sorted by their
dominant latent dimension.

In [ ]:
# Loading matrix from the final fitted GPFA model
C = final_gpfa.params_estimated["C"]

dominant_latent = np.argmax(np.abs(C), axis=1)
order = np.argsort(dominant_latent)
C_sorted = C[order]

plt.figure(figsize=(8,10))
plt.imshow(C_sorted, aspect="auto",  cmap="bwr",  interpolation="nearest")
plt.colorbar( label="Loading weight")
plt.xlabel("Latent dimension")
plt.ylabel("Neuron (sorted)")
plt.title("GPFA loading matrix sorted by dominant latent")
plt.tight_layout()
plt.show()

### 11.6 Trial-averaged latent dynamics (per-dimension subplots)

Same trial-averaged mean ± SEM as Section 11.3, shown as one subplot per latent
dimension instead of overlaid — useful when there are many latent dimensions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

latent_array = np.stack(latent_trajectories, axis=0)
# shape: trials x latent_dims x time
print(latent_array.shape)

mean_latent = np.mean(latent_array, axis=0)
sem_latent = (np.std(latent_array, axis=0, ddof=1) / np.sqrt(latent_array.shape[0]))
n_latents = mean_latent.shape[0]
time_bins = np.arange(mean_latent.shape[1])
fig, axes = plt.subplots(n_latents, 1, figsize=(10, 2*n_latents), sharex=True)

for dim in range(n_latents):
    axes[dim].plot(time_bins, mean_latent[dim])
    axes[dim].fill_between(time_bins, mean_latent[dim]-sem_latent[dim], mean_latent[dim]+sem_latent[dim], alpha=0.3)
    axes[dim].set_ylabel(f"L{dim+1}")
    axes[dim].grid(alpha=0.3)
axes[-1].set_xlabel("Time bin")
plt.suptitle("Trial-averaged GPFA latent dynamics")
plt.tight_layout()
plt.show()

### 11.7 Latent dimension variance / effective dimensionality

Quantify how much variance each latent dimension carries (across trials and time),
and plot the cumulative variance explained as dimensions are added in order of
decreasing variance — a quick check on how many dimensions are actually "doing
work" in the fitted model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# trials x latent_dims x time
latent_array = np.stack(latent_trajectories, axis=0)

print("Latent array shape:", latent_array.shape)

# Variance of each latent dimension across trials and time
latent_variance = np.var(
    latent_array,
    axis=(0, 2)
)

print("Variance per latent dimension:")
for i, v in enumerate(latent_variance):
    print(f"Latent {i+1}: {v:.4f}")


# Plot
plt.figure(figsize=(8,4))

plt.bar(
    np.arange(1, len(latent_variance)+1),
    latent_variance
)

plt.xlabel("Latent dimension")
plt.ylabel("Variance")
plt.title(
    f"GPFA latent dimension variance (x_dim={latent_array.shape[1]})"
)

plt.xticks(
    np.arange(1, len(latent_variance)+1)
)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

sorted_variance = np.sort(latent_variance)[::-1]

variance_fraction = sorted_variance / np.sum(sorted_variance)
cumulative = np.cumsum(variance_fraction)

plt.plot(
    np.arange(1, len(cumulative)+1),
    cumulative,
    marker="o"
)

plt.xlabel("Number of latent dimensions")
plt.ylabel("Cumulative variance")
plt.title("GPFA effective dimensionality (sorted)")
plt.grid(alpha=0.3)
plt.show()

## 12. Relate Latent Dimensions to Behavior

For each GPFA latent dimension, compute its Pearson correlation with each
behavioral variable (flattened across trials and time bins), to check whether
population-level neural dynamics along that dimension track any measured
behavior.

> **Requires:** a `binned_behaviors` dict mapping behavior name → array of shape
> `(trials, time bins)`, aligned to the same trial windows and bin size as
> `binned_trials` above. This notebook doesn't build `binned_behaviors` itself —
> load or construct it (e.g. from this session's behavioral traces) before
> running this cell.

In [ ]:
'''
When the neural population moves along a particular GPFA latent dimension, does that movement track any behavioral variable?
Each number is a Pearson correlation coefficient:
+1 → latent increases when behavior increases
-1 → latent decreases when behavior increases
0 → no linear relationship
'''

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Stack latent trajectories
latent_array = np.stack(latent_trajectories)
# Shape: trials x latent_dims x time
print("Latent shape:", latent_array.shape)

# Choose behaviors to analyze
behavior_names = [
    "Lick rate",
    "Wheel",
    "Whisking",
    "Pupil area",
    "Pupil radius",
    "Respiration frequency",
    "Respiration amplitude",
    "Face motion"
]
n_latents = latent_array.shape[1]

# Store correlations
behavior_corr = np.zeros((n_latents, len(behavior_names)))

# Calculate correlations
for latent_dim in range(n_latents):
    # flatten latent across trials and time
    latent_values = latent_array[:, latent_dim, :].flatten()
    for j, behavior_name in enumerate(behavior_names):
        behavior_values = binned_behaviors[behavior_name].flatten()
        r, p = pearsonr(latent_values, behavior_values)
        behavior_corr[latent_dim, j] = r

# Print table
print("\nLatent-behavior correlations")
for i in range(n_latents):
    print(f"Latent {i+1}:", {behavior_names[j]: round(behavior_corr[i,j],3) for j in range(len(behavior_names))})

# Plot heatmap
plt.figure(figsize=(10,8))
plt.imshow(behavior_corr, aspect="auto", cmap="bwr", vmin=-1, vmax=1)
plt.colorbar(label="Pearson r")
plt.xticks(np.arange(len(behavior_names)), behavior_names, rotation=45, ha="right")
plt.yticks(np.arange(n_latents), [f"Latent {i+1}" for i in range(n_latents)])
plt.xlabel("Behavioral variable")
plt.ylabel("GPFA latent dimension")
plt.title("GPFA latent dimensions correlated with behavior")
plt.tight_layout()
plt.show()